# Fine-Tuning: LoRA, QLoRA-style Quantization, SFT Masking, DPO and RLHF Objectives

We implement the core mechanics from scratch on toy models: why LoRA trains so few parameters, why merging is
exact, how 4-bit blockwise quantization of a frozen base works, how SFT masks prompt tokens, and what the DPO
and RLHF losses compute. These are teaching implementations; use maintained libraries for real training.

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('06-generative-ai/fine-tuning/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
import peft_lib
print('peft_lib loaded')

## 1. LoRA: low-rank updates on a frozen model
$W' = W + \frac{\alpha}{r} BA$ with $B=0$ at init, so the adapted model starts identical to the base.

In [ ]:
import torch, torch.nn as nn
from peft_lib import *
torch.manual_seed(0)

class Toy(nn.Module):
    def __init__(self):
        super().__init__()
        self.q_proj, self.v_proj, self.out = nn.Linear(64, 64), nn.Linear(64, 64), nn.Linear(64, 8)
    def forward(self, x):
        return self.out(torch.relu(self.q_proj(x)) + self.v_proj(x))

m = Toy()
t0, total0 = count_params(m)
apply_lora(m, {'q_proj', 'v_proj'}, r=4, alpha=8)
t1, total1 = count_params(m)
print(f'before: {t0:,} trainable / {total0:,}   after LoRA: {t1:,} trainable / {total1:,} ({100*t1/total1:.1f}%)')

In [ ]:
x, y = torch.randn(256, 64), torch.randn(256, 8)
frozen_before = m.out.weight.clone()
opt = torch.optim.Adam([p for p in m.parameters() if p.requires_grad], lr=1e-2)
for step in range(101):
    opt.zero_grad(); loss = nn.functional.mse_loss(m(x), y); loss.backward(); opt.step()
    if step % 25 == 0: print(f'step {step:3d} loss {loss.item():.4f}')
print('frozen layer unchanged:', torch.equal(frozen_before, m.out.weight))
out_lora = m(x).detach()
merge_lora(m)
print('merged model matches adapter model:', torch.allclose(out_lora, m(x).detach(), atol=1e-4))

## 2. Quantizing the frozen base (QLoRA idea)
Blockwise absmax scaling to 4 bits. Levels placed at normal-distribution quantiles (NormalFloat-*style*) beat uniform levels on Gaussian-like weights.

In [ ]:
torch.manual_seed(1)
w = torch.randn(256, 256)
for name, lv in [('uniform 4-bit', uniform_levels()), ('NF-style 4-bit', nf_style_levels())]:
    codes, scales, shape = blockwise_quantize(w, lv, block=64)
    rec = blockwise_dequantize(codes, scales, shape, lv)
    print(f'{name:15s} MSE={((rec-w)**2).mean():.5f}')
q = QLoRALinear(nn.Linear(256, 256), r=8)
qb, fb = q.stored_bytes()
print(f'stored ~{qb/1024:.1f} KiB vs fp16 {fb/1024:.1f} KiB  ({fb/qb:.1f}x smaller); trainable: {sum(p.numel() for p in q.parameters() if p.requires_grad):,}')

## 3. Supervised fine-tuning: loss only on the response
Prompt tokens get label `-100` so they don't contribute gradient.

In [ ]:
ids, labels = build_sft_example(prompt_ids=[11, 12, 13], response_ids=[21, 22])
print('ids   :', ids); print('labels:', labels)

## 4. Preference optimization: DPO and reward modelling
DPO increases the policy's relative log-likelihood of the chosen response vs. a frozen reference, with no separate reward model or RL loop.

In [ ]:
import math
z = torch.zeros(1)
print('policy == reference  ->', round(float(dpo_loss(z, z, z, z)), 4), '(= ln 2 =', round(math.log(2), 4), ')')
for m_ in [-4, -1, 0, 1, 4]:
    print(f'chosen-vs-rejected margin {m_:+d} -> DPO loss {float(dpo_loss(z+m_/2, z-m_/2, z, z, beta=0.5)):.4f}')
print('reward-model loss (chosen scores higher):', round(float(reward_model_loss(torch.tensor([2.0]), torch.tensor([0.0]))), 4))
print('KL-shaped RLHF reward:', float(kl_shaped_reward(torch.tensor([1.0]), torch.tensor([-1.0]), torch.tensor([-3.0]), beta=0.5)))